# Fine-tune `microsoft/trocr-base-printed` on receipt words (Colab)

Runtime → Change runtime type → **GPU** before running. On a CPU the notebook runs a tiny smoke test instead.

`notebooks/eda_words.ipynb` found two problems with the first recognizer (ViT → BERT, character error rate 0.71):
its cross-attention (the only link from image to text) never trained, and the data is heavily skewed (`1` is 8% of
words, `0` is 17% of characters), so predicting frequent patterns lowered the loss without reading the image.

This notebook addresses both:

| Problem | Fix here |
|---|---|
| Cross-attention starts random and barely trains | Start from `microsoft/trocr-base-printed`: encoder, decoder **and cross-attention** already trained to read printed receipts (fine-tuned on SROIE) |
| One learning rate for everything, long fixed warmup | Separate learning rates for encoder and decoder; warmup is 10% of the steps |
| Frequent words dominate each epoch | **Weighted sampling**: down-weights frequent words, up-weights words with rare characters |
| Few examples of rare characters and prices | **Synthetic words** rendered on the fly (prices, quantities, rare characters), 30% on top of the real words |
| Loss can drop without reading | Checkpoint picked on **validation CER**, plus a **blind test** each epoch (CER on white images vs real crops) and CER per class (numbers vs words) |

The split is the same as in training and the EDA (`split_receipts`, seed 42), so results are comparable. The best
checkpoint goes to `MyDrive/document-processing/checkpoints/trocr-printed/best`; copy it to `checkpoints/` locally and run
`make pipeline OCR=model RECOGNIZER=checkpoints/trocr-printed/best`.

## Setup

In [ ]:
import os, subprocess, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run(["nvidia-smi"])

In [ ]:
# Colab: mount Drive, clone or update the repo, install what the notebook needs, copy the data to local disk.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DATA = "/content/drive/MyDrive/dataset_receipt"
    OUT_DIR = "/content/drive/MyDrive/document-processing/checkpoints/trocr-printed"  # on Drive: survives disconnects

    try:  # for a private repo, add a GITHUB_TOKEN secret in the Colab sidebar (key icon)
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    REPO_DIR = "/content/document-processing"
    repo_url = f"https://{token + '@' if token else ''}github.com/alexisvannson/document-processing.git"
    if os.path.isdir(REPO_DIR):
        subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", repo_url, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    # Colab ships torch, transformers, opencv, pandas and matplotlib; ocr/dataset.py also needs these
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "albumentations>=2.0", "scikit-image",
                    "pyclipper", "shapely"], check=True)
    # reading images from Drive every epoch is slow: copy them to local disk once
    subprocess.run(["rsync", "-a", f"{DRIVE_DATA}/", "/content/dataset_receipt/"], check=True)
    DATA_DIR = "/content/dataset_receipt"
else:
    if not os.path.isdir("dataset_receipt") and os.path.isdir("../dataset_receipt"):
        os.chdir("..")
    DATA_DIR, OUT_DIR = "dataset_receipt", "checkpoints/trocr-printed"
print("repo:", os.getcwd(), "| data:", DATA_DIR, "| checkpoints:", OUT_DIR)

## Configuration

In [ ]:
import torch

SMOKE = not torch.cuda.is_available()  # on CPU: a few steps on a handful of words, to check the code runs
SAVE = not SMOKE                       # a smoke test doesn't write a 1.3 GB checkpoint

MODEL_NAME = "microsoft/trocr-base-printed"
SEED = 42                 # same seed as training and the EDA: same train / val / test receipts
EPOCHS = 1 if SMOKE else 15
PATIENCE = 4              # stop when validation CER hasn't improved for this many epochs
BATCH_SIZE = 4 if SMOKE else 16
LR_ENCODER = 2e-5         # pretrained image encoder: small steps
LR_DECODER = 5e-5         # decoder and cross-attention
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.1
SYNTHETIC_FRAC = 0.3      # synthetic words per epoch, as a fraction of the real training words
MAX_TARGET_TOKENS = 32
BLIND_SAMPLES = 8 if SMOKE else 200   # val crops re-read from a white image each epoch
TEST_BEAMS = 4            # beam search for the final test score (greedy during training, for speed)
USE_WANDB = False         # set True and add a WANDB_API_KEY secret to log to Weights & Biases
print(f"smoke test: {SMOKE} | epochs {EPOCHS} | batch {BATCH_SIZE}")

## Data: word crops from the ground-truth boxes

In [ ]:
import collections, math, random, re, time, warnings
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", message="IProgress not found")
from transformers import (AutoTokenizer, TrOCRProcessor, ViTImageProcessor, VisionEncoderDecoderModel,
                          get_cosine_schedule_with_warmup)
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()

from ocr.dataset import crop_quad, get_recognition_transforms, split_receipts
from training.train_trocr import edit_distance

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def word_crops(df):
    crops, texts = [], []
    for row in df.itertuples():
        image = cv2.cvtColor(cv2.imread(os.path.join(DATA_DIR, "images", row.file_name)), cv2.COLOR_BGR2RGB)
        for box, word in zip(row.bboxes, row.words):
            crops.append(crop_quad(image, [[box[f"x{k}"], box[f"y{k}"]] for k in range(1, 5)]))
            texts.append(word)
    return crops, texts

train_df, val_df, test_df = split_receipts(pd.read_pickle(os.path.join(DATA_DIR, "metadata.pkl")), seed=SEED)
if SMOKE:
    train_df, val_df, test_df = train_df.head(2), val_df.head(1), test_df.head(1)
train_crops, train_texts = word_crops(train_df)
val_crops, val_texts = word_crops(val_df)
test_crops, test_texts = word_crops(test_df)
print(f"words: train {len(train_texts):,}, val {len(val_texts):,}, test {len(test_texts):,}")

## Fix 1: weighted sampling

Each word's sampling weight is `1 / sqrt(frequency)`, so `1` (the most frequent word) is drawn far less often than
its raw count. Words containing a rare character get up to 3× more weight. Sampling is with replacement, and each
epoch draws as many real words as there are in the training set.

In [ ]:
word_freq = collections.Counter(train_texts)
char_freq = collections.Counter(c for w in train_texts for c in w)
median_char = float(np.median(list(char_freq.values())))

def sample_weight(word):
    rarity = max((median_char / char_freq[c]) ** 0.5 for c in word) if word else 1.0
    return (1 / math.sqrt(word_freq[word])) * min(max(rarity, 1.0), 3.0)

weights = np.array([sample_weight(w) for w in train_texts])
probs = weights / weights.sum()

def expected_share(select, p):
    return sum(p_i for w, p_i in zip(train_texts, p) if select(w))

uniform = np.full(len(train_texts), 1 / len(train_texts))
def char_share(ch, p):
    total = sum(p_i * len(w) for w, p_i in zip(train_texts, p))
    return sum(p_i * w.count(ch) for w, p_i in zip(train_texts, p)) / total
rare_chars = {c for c, n in char_freq.items() if n < 10}
has_rare = lambda w: bool(set(w) & rare_chars)
comparison = pd.DataFrame({
    "raw data": [expected_share(lambda w: w == "1", uniform), char_share("0", uniform), expected_share(has_rare, uniform)],
    "sampled": [expected_share(lambda w: w == "1", probs), char_share("0", probs), expected_share(has_rare, probs)],
}, index=["word '1' (share of words)", "character '0' (share of characters)", "words with a rare character"])
print(comparison.map("{:.1%}".format).to_string())

## Fix 2: synthetic words

Rendered on the fly with the DejaVu fonts that ship with matplotlib: prices with `,` or `.` separators,
quantities and percentages, training words in random case, and random strings biased towards rare characters.
They go through the same augmentations as real crops.

In [ ]:
import matplotlib
from PIL import Image, ImageDraw, ImageFont

FONT_DIR = os.path.join(matplotlib.get_data_path(), "fonts", "ttf")
FONTS = [os.path.join(FONT_DIR, f) for f in ("DejaVuSans.ttf", "DejaVuSans-Bold.ttf", "DejaVuSansMono.ttf",
                                             "DejaVuSansMono-Bold.ttf", "DejaVuSerif.ttf")]
vocab = sorted(word_freq)
charset = sorted(c for c in char_freq if not c.isspace())
char_p = np.array([1 / math.sqrt(char_freq[c]) for c in charset]); char_p /= char_p.sum()

def synthetic_text(rng):
    kind = rng.random()
    if kind < 0.35:   # a price: 1,500 / 23.000 / 125,000.00
        value = int(round(10 ** rng.uniform(2, 6.5), -2))
        text = f"{value:,}".replace(",", rng.choice([",", "."]))
        return text + (".00" if rng.random() < 0.1 else "")
    if kind < 0.50:   # a quantity or percentage: 2 / 3x / 10%
        n = int(rng.integers(0, 25))
        return rng.choice([str(n), f"{n}x", f"{n}%", f"x{n}"])
    if kind < 0.80:   # a real word, in a random case
        w = vocab[rng.integers(len(vocab))]
        return rng.choice([w, w.upper(), w.lower(), w.title()])
    return "".join(rng.choice(charset, size=int(rng.integers(2, 9)), p=char_p))  # rare characters

def render(text, rng):
    font = ImageFont.truetype(FONTS[rng.integers(len(FONTS))], int(rng.integers(26, 46)))
    left, top, right, bottom = font.getbbox(text)
    pad = int(rng.integers(4, 14))
    paper = int(rng.integers(190, 256))
    img = Image.new("RGB", (right - left + 2 * pad, bottom - top + 2 * pad), (paper, paper, int(paper * rng.uniform(0.9, 1.0))))
    ink = int(rng.integers(0, 80))
    ImageDraw.Draw(img).text((pad - left, pad - top), text, font=font, fill=(ink, ink, ink))
    return np.asarray(img)

rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(3, 4, figsize=(12, 3.6))
for ax in axes.flat:
    t = synthetic_text(rng); ax.imshow(render(t, rng)); ax.set_title(repr(t), fontsize=9); ax.axis("off")
plt.suptitle("Synthetic words (before augmentation)"); plt.tight_layout(); plt.show()

## Model and data loaders

In [ ]:
# The checkpoint ships only vocab.json + merges.txt, which transformers 5 can no longer convert to a fast tokenizer
# ("Couldn't instantiate the backend tokenizer"). They are RoBERTa's: the same 50,265 entries and identical merges,
# and RoBERTa ships a ready-made tokenizer.json, so loading it from there is exactly equivalent.
try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
except ValueError:
    tokenizer = AutoTokenizer.from_pretrained("FacebookAI/roberta-large")
processor = TrOCRProcessor(image_processor=ViTImageProcessor.from_pretrained(MODEL_NAME), tokenizer=tokenizer)
model = VisionEncoderDecoderModel.from_pretrained(MODEL_NAME)
# Train with the same token conventions the model generates with: decoder start </s> (2), then the text
# tokens, then </s>, with no <s> (0), even though the tokenizer adds one by default.
model.config.decoder_start_token_id = model.generation_config.decoder_start_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.generation_config.max_new_tokens = MAX_TARGET_TOKENS
model.to(device)
def encode(text):
    ids = tokenizer(text, add_special_tokens=False, max_length=MAX_TARGET_TOKENS - 1, truncation=True).input_ids
    return ids + [tokenizer.eos_token_id]

print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f}M parameters | decoder start token "
      f"{model.config.decoder_start_token_id} | labels for '16,500': {encode('16,500')}")

train_tf, eval_tf = get_recognition_transforms(model.config.encoder.image_size)  # = the TrOCR processor + augmentation

class Words(torch.utils.data.Dataset):
    """Real crops first, then n_synthetic freshly rendered words (a different set every epoch)."""
    def __init__(self, crops, texts, transform, n_synthetic=0):
        self.crops, self.texts, self.transform, self.n_synthetic, self.epoch = crops, texts, transform, n_synthetic, 0
    def __len__(self):
        return len(self.crops) + self.n_synthetic
    def __getitem__(self, i):
        if i < len(self.crops):
            image, text = self.crops[i], self.texts[i]
        else:
            rng = np.random.default_rng((SEED, self.epoch, i))
            text = synthetic_text(rng); image = render(text, rng)
        return self.transform(image=image)["image"], torch.tensor(encode(text))

def collate(batch):
    images, labels = zip(*batch)
    padded = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)  # -100: ignored by the loss
    return torch.stack(images), padded

class EpochSampler(torch.utils.data.Sampler):
    """Each epoch: len(real) real indices drawn by weight, plus every synthetic index, shuffled together."""
    def __init__(self, dataset, probs):
        self.dataset, self.probs = dataset, torch.tensor(probs)
    def __len__(self):
        return len(self.dataset)
    def __iter__(self):
        g = torch.Generator().manual_seed(SEED + self.dataset.epoch)
        n_real = len(self.dataset.crops)
        real = torch.multinomial(self.probs, n_real, replacement=True, generator=g)
        synthetic = torch.arange(n_real, len(self.dataset))
        order = torch.cat([real, synthetic])[torch.randperm(len(self.dataset), generator=g)]
        return iter(order.tolist())

n_synthetic = 16 if SMOKE else int(SYNTHETIC_FRAC * len(train_texts))
train_set = Words(train_crops, train_texts, train_tf, n_synthetic)
workers = 0 if SMOKE else 2
train_loader = torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE, sampler=EpochSampler(train_set, probs),
                                           collate_fn=collate, num_workers=workers)
print(f"per epoch: {len(train_texts):,} sampled real + {n_synthetic:,} synthetic words = {len(train_loader)} steps")

## Evaluation: CER overall and per class, plus the blind test

`reading gap` = CER on white images − CER on the real crops. A model that reads has a large gap; the first checkpoint
had almost none. For reference, zero-shot `trocr-base-printed` scored CER 0.251 (62% exact) on the first 80 validation
words, against 0.71 (7%) for the first checkpoint; most of its errors are receipt conventions (`Subtotal` → `SUBTOTAL`,
`13,636` → `13.636`) that fine-tuning should fix.

In [ ]:
AMOUNT = re.compile(r"^[-(]?(rp\.?\s*)?@?\d{1,3}([.,]\d{3})+([.,]\d{2})?\)?$|^\d+$", re.I)
def word_class(w):
    return "number" if AMOUNT.match(w.strip(":")) else "word" if re.fullmatch(r"[A-Za-z]+[.:]?", w) else "other"

def cer(preds, truths):
    return sum(edit_distance(p, t) for p, t in zip(preds, truths)) / max(sum(len(t) for t in truths), 1)

@torch.no_grad()
def read(crops, beams=1, batch_size=32):
    model.eval()
    preds = []
    for i in range(0, len(crops), batch_size):
        batch = torch.stack([eval_tf(image=c)["image"] for c in crops[i:i + batch_size]]).to(device)
        out = model.generate(pixel_values=batch, num_beams=beams, max_new_tokens=MAX_TARGET_TOKENS)
        preds += [p.strip() for p in tokenizer.batch_decode(out, skip_special_tokens=True)]
    return preds

def evaluate(crops, texts, beams=1):
    preds = read(crops, beams)
    blind = read([np.full_like(c, 255) for c in crops[:BLIND_SAMPLES]], beams)
    result = {"cer": cer(preds, texts), "word_acc": float(np.mean([p == t for p, t in zip(preds, texts)])),
              "blind_cer": cer(blind, texts[:BLIND_SAMPLES])}
    result["reading_gap"] = result["blind_cer"] - cer(preds[:BLIND_SAMPLES], texts[:BLIND_SAMPLES])
    for cls in ("number", "word", "other"):
        idx = [i for i, t in enumerate(texts) if word_class(t) == cls]
        result[f"cer_{cls}"] = cer([preds[i] for i in idx], [texts[i] for i in idx]) if idx else float("nan")
    return result, preds

def show(name, r):
    print(f"{name}: CER {r['cer']:.3f} | exact {r['word_acc']:.2f} | numbers {r['cer_number']:.3f} | "
          f"words {r['cer_word']:.3f} | blind CER {r['blind_cer']:.3f} (reading gap {r['reading_gap']:+.3f})")

zero_shot, _ = evaluate(val_crops, val_texts)
show("zero-shot, before fine-tuning", zero_shot)

## Fine-tune

In [ ]:
encoder_params = list(model.encoder.parameters())
decoder_params = list(model.decoder.parameters())  # includes the cross-attention
optimizer = torch.optim.AdamW([{"params": encoder_params, "lr": LR_ENCODER},
                               {"params": decoder_params, "lr": LR_DECODER}], weight_decay=WEIGHT_DECAY)
total_steps = EPOCHS * len(train_loader)
scheduler = get_cosine_schedule_with_warmup(optimizer, int(WARMUP_FRAC * total_steps), total_steps)

amp_dtype = None
if device.type == "cuda":
    amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
scaler = torch.amp.GradScaler("cuda") if amp_dtype == torch.float16 else None

run = None
if USE_WANDB:
    import wandb
    if IN_COLAB:
        from google.colab import userdata
        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    run = wandb.init(project="document-processing", job_type="recognition-trocr-printed",
                     config={k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, bool))})

best_cer, stale, history = float("inf"), 0, []
for epoch in range(1, EPOCHS + 1):
    model.train(); train_set.epoch = epoch
    start, total_loss = time.time(), 0.0
    for step, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        with torch.autocast(device.type, dtype=amp_dtype, enabled=amp_dtype is not None):
            loss = model(pixel_values=images, labels=labels).loss
        optimizer.zero_grad()
        if scaler:
            scaler.scale(loss).backward(); scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        if SMOKE and step == 2:
            break
    val, _ = evaluate(val_crops, val_texts)
    history.append({"epoch": epoch, "train_loss": total_loss / (step + 1), **val})
    print(f"epoch {epoch} | loss {total_loss / (step + 1):.3f} | {time.time() - start:.0f}s")
    show("  val", val)
    if run:
        run.log({"train/loss": total_loss / (step + 1), **{f"val/{k}": v for k, v in val.items()}}, step=epoch)
    if val["cer"] < best_cer:
        best_cer, stale = val["cer"], 0
        if SAVE:
            model.save_pretrained(os.path.join(OUT_DIR, "best")); processor.save_pretrained(os.path.join(OUT_DIR, "best"))
        print(f"  new best val CER {best_cer:.3f}" + (f" -> {OUT_DIR}/best" if SAVE else " (not saved: smoke test)"))
    else:
        stale += 1
        if stale >= PATIENCE:
            print(f"  no improvement for {PATIENCE} epochs: stopping"); break

## Test: the held-out receipts, scored once with the best checkpoint

In [ ]:
if SAVE:
    model = VisionEncoderDecoderModel.from_pretrained(os.path.join(OUT_DIR, "best")).to(device)
test, test_preds = evaluate(test_crops, test_texts, beams=TEST_BEAMS)
show(f"test (beam {TEST_BEAMS})", test)
if run:
    run.summary.update({f"test/{k}": v for k, v in test.items()}); run.finish()

pd.DataFrame(history).set_index("epoch")[["cer", "blind_cer", "cer_number", "cer_word"]].plot(
    marker="o", figsize=(8, 3.5), title="Validation CER per epoch (blind_cer: white images)")
plt.show()

errors = [(t, p) for t, p in zip(test_texts, test_preds) if t != p]
print(f"{len(errors)} of {len(test_texts)} test words wrong; a sample:")
print(pd.DataFrame(errors[:25], columns=["truth", "prediction"]).to_string(index=False))

## Use it in the pipeline

Download `MyDrive/document-processing/checkpoints/trocr-printed/best` to `checkpoints/trocr-printed/best` in the repo,
then:

```bash
make pipeline OCR=model RECOGNIZER=checkpoints/trocr-printed/best
make dbt
```

`ocr/inference.py` detects the TrOCR-style checkpoint (it has `tokenizer.json` or `vocab.json`) and uses its
tokenizer. Compare the
published and quarantined counts with the ground-truth OCR run (139 / 38): the difference is the error OCR adds.